#HR Attrition SQL Analysis
This notebook performs SQL-based business analysis on the IBM HR Analytics Employee Attrition dataset using SQLite.

#Project Objective


*  Analyze employee attrition patterns
*   Identify major drivers of workforce turnover

*   Evaluate the impact of salary, overtime, tenure, and satisfaction
*   Estimate the financial impact of attrition

*   Identify high-risk employee groups of retention strategies








#Database Setup
The HR Attrition dataset is loaded into an SQLite database to simulate a real-world relational database analysis workflow commonly used in business environments.

In [6]:
import sqlite3, pandas as pd

df = pd.read_csv('HR-Employee-Attrition.csv')

conn = sqlite3.connect('hr_attrition.db')

df.to_sql('employees', conn, if_exists='replace', index=False)

print("Database ready.")

Database ready.


In [7]:
query_1 = """ -- ============================================================
-- SECTION 1: OVERVIEW
-- ============================================================

-- 1.1 Total headcount and overall attrition rate
SELECT
    COUNT(*)                                                        AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)             AS employees_left,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)             AS employees_stayed,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 2
    )                                                               AS attrition_rate_pct
FROM employees;
 """

In [8]:
df1 = pd.read_sql_query(query_1, conn)
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,1,2061,...,3,80,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,1,2062,...,1,80,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2064,...,2,80,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,1,2065,...,4,80,0,17,3,2,9,6,0,8


In [9]:
query_2 = """-- 1.2 Average monthly income: left vs stayed
SELECT
    Attrition,
    ROUND(AVG(MonthlyIncome), 2)    AS avg_monthly_income,
    ROUND(MIN(MonthlyIncome), 2)    AS min_income,
    ROUND(MAX(MonthlyIncome), 2)    AS max_income,
    COUNT(*)                        AS headcount
FROM employees
GROUP BY Attrition;
"""
df = pd.read_sql_query(query_2, conn)
df1

,total_employees,employees_left,employees_stayed,attrition_rate_pct
0,1470,237,1233,16.12


# Section 1 — Employee Attrition Overview

This section provides a high-level overview of the workforce, including total employees, overall attrition rate, and salary comparison between employees who stayed and employees who left.

## 1.1 Total Workforce & Attrition Rate

This query calculates:

* Total employee count
* Employees who left the organization
* Employees who stayed
* Overall attrition percentage

## 1.2 Salary Comparison: Employees Who Left vs Stayed

This analysis compares salary distribution between retained and attrited employees to evaluate whether compensation may influence turnover.



# SECTION 2: DEPARTMENT ANALYSIS


In [10]:
query_3 = """ -- 2.1 Attrition RATE by Department (use rate, not just count)
SELECT
    Department,
    COUNT(*)                                                            AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                 AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 2
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC; """
df = pd.read_sql_query(query_3, conn)
df

,Department,total_employees,employees_left,attrition_rate_pct
0,Sales,446,92,20.63
1,Human Resources,63,12,19.05
2,Research & Development,961,133,13.84


In [11]:
query_4 = """ -- 2.2 Job Role attrition rate — ranked
SELECT
    JobRole,
    COUNT(*)                                                            AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                 AS left_count,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY JobRole
ORDER BY attrition_rate_pct DESC; """

df = pd.read_sql_query(query_4, conn)
df

,JobRole,total,left_count,attrition_rate_pct
0,Sales Representative,83,33,39.8
1,Laboratory Technician,259,62,23.9
2,Human Resources,52,12,23.1
3,Sales Executive,326,57,17.5
4,Research Scientist,292,47,16.1
5,Manufacturing Director,145,10,6.9
6,Healthcare Representative,131,9,6.9
7,Manager,102,5,4.9
8,Research Director,80,2,2.5


# Section 2 — Department & Job Role Analysis

This section identifies departments and job roles with the highest attrition rates to determine where employee turnover is most concentrated.

## 2.1 Attrition Rate by Department

This query evaluates employee attrition across organizational departments.

### Business Insight

Departments with higher attrition rates may indicate operational stress, poor management conditions, or reduced employee satisfaction.

## 2.2 Job Role Attrition Ranking

This analysis ranks job roles based on attrition rate to identify the most vulnerable positions.

### Business Insight

Certain job roles may experience higher turnover due to workload, career stagnation, or compensation-related factors.


#SECTION 3: OVERTIME ANALYSIS

In [12]:
query_5 = """SELECT
    OverTime,
    COUNT(*)                                                            AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                 AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 2
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY OverTime
ORDER BY attrition_rate_pct DESC;"""

df = pd.read_sql_query(query_5, conn)
df

,OverTime,total_employees,employees_left,attrition_rate_pct
0,Yes,416,127,30.53
1,No,1054,110,10.44


In [13]:
query_6 = """ -- 3.2 Overtime + Department combo (where is burnout worst?)
SELECT
    Department,
    OverTime,
    COUNT(*)                                                            AS total,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY Department, OverTime
ORDER BY Department, attrition_rate_pct DESC;"""

df = pd.read_sql_query(query_6, conn)
df

,Department,OverTime,total,attrition_rate_pct
0,Human Resources,Yes,17,29.4
1,Human Resources,No,46,15.2
2,Research & Development,Yes,271,27.3
3,Research & Development,No,690,8.6
4,Sales,Yes,128,37.5
5,Sales,No,318,13.8


# Section 3 — Overtime & Burnout Analysis

This section analyzes whether overtime work contributes to increased employee attrition.

## 3.1 Attrition Rate by Overtime Status

This query compares attrition rates between employees who work overtime and those who do not.

### Business Insight

Employees working overtime are more likely to leave the organization, suggesting workload and burnout are major retention risks.

## 3.2 Department-wise Burnout Risk Analysis

This analysis combines overtime and department data to identify where burnout-related attrition is highest.

### Business Insight

Departments with both high overtime and high attrition may require workload balancing and employee wellness interventions.


#SECTION 4: SALARY & COMPENSATION

In [14]:
query_7 = """ -- 4.1 Attrition rate by salary band
SELECT
    CASE
        WHEN MonthlyIncome < 3000  THEN '1. Low      (<$3k)'
        WHEN MonthlyIncome < 6000  THEN '2. Mid      ($3k-$6k)'
        WHEN MonthlyIncome < 10000 THEN '3. High     ($6k-$10k)'
        ELSE                            '4. Very High (>$10k)'
    END                                                                 AS salary_band,
    COUNT(*)                                                            AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                 AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY salary_band
ORDER BY salary_band;"""

df = pd.read_sql_query(query_7, conn)
df


,salary_band,total_employees,employees_left,attrition_rate_pct
0,1. Low (<$3k),395,113,28.6
1,2. Mid ($3k-$6k),519,66,12.7
2,3. High ($6k-$10k),275,33,12.0
3,4. Very High (>$10k),281,25,8.9


In [15]:
query_8 = """ -- 4.2 Salary + Overtime combo (underpaid AND overworked = highest risk)
SELECT
    CASE
        WHEN MonthlyIncome < 3000  THEN 'Low'
        WHEN MonthlyIncome < 6000  THEN 'Mid'
        WHEN MonthlyIncome < 10000 THEN 'High'
        ELSE                            'Very High'
    END                                                                 AS salary_band,
    OverTime,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct,
    COUNT(*)                                                            AS total
FROM employees
GROUP BY salary_band, OverTime
ORDER BY attrition_rate_pct DESC;"""

df = pd.read_sql_query(query_8, conn)
df

,salary_band,OverTime,attrition_rate_pct,total
0,Low,Yes,56.1,114
1,Mid,Yes,24.1,141
2,High,Yes,22.4,76
3,Low,No,17.4,281
4,Very High,Yes,14.1,85
5,Mid,No,8.5,378
6,High,No,8.0,199
7,Very High,No,6.6,196


# Section 4 — Salary & Compensation Analysis

This section examines the relationship between employee compensation and attrition.

## 4.1 Attrition by Salary Band

This query groups employees into salary categories and compares attrition across compensation levels.

### Business Insight

Employees in lower salary bands experience higher attrition, indicating compensation may be a major retention factor.

## 4.2 Salary + Overtime Risk Combination

This analysis evaluates attrition among employees who are both underpaid and overworked.

### Business Insight

The combination of low salary and overtime creates one of the highest-risk attrition profiles within the organization.


#SECTION 5: TENURE ANALYSIS

In [16]:
query_9 = """ -- 5.1 Attrition rate by tenure bucket
SELECT
    CASE
        WHEN YearsAtCompany <= 2  THEN '1. 0-2 years  (danger zone)'
        WHEN YearsAtCompany <= 5  THEN '2. 3-5 years'
        WHEN YearsAtCompany <= 10 THEN '3. 6-10 years'
        ELSE                           '4. 10+ years  (stable)'
    END                                                                 AS tenure_bucket,
    COUNT(*)                                                            AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                 AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY tenure_bucket
ORDER BY tenure_bucket;"""

df = pd.read_sql_query(query_9, conn)
df



,tenure_bucket,total_employees,employees_left,attrition_rate_pct
0,1. 0-2 years (danger zone),342,102,29.8
1,2. 3-5 years,434,60,13.8
2,3. 6-10 years,448,55,12.3
3,4. 10+ years (stable),246,20,8.1


In [17]:
query_10 = """ -- 5.2 Years since last promotion vs attrition
SELECT
    YearsSinceLastPromotion,
    COUNT(*)                                                            AS total,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY YearsSinceLastPromotion
ORDER BY YearsSinceLastPromotion;"""

df = pd.read_sql_query(query_10, conn)
df

,YearsSinceLastPromotion,total,attrition_rate_pct
0,0,581,18.9
1,1,357,13.7
2,2,159,17.0
3,3,52,17.3
4,4,61,8.2
5,5,45,4.4
6,6,32,18.8
7,7,76,21.1
8,8,18,0.0
9,9,17,23.5


# Section 5 — Employee Tenure Analysis

This section analyzes how employee experience and promotion history influence attrition.

## 5.1 Attrition by Tenure Group

This query evaluates employee attrition across different tenure ranges.

### Business Insight

Employees within their first few years at the company show the highest attrition, highlighting challenges in early-stage retention.

## 5.2 Promotion Delay vs Attrition

This analysis examines whether delayed promotions contribute to employee turnover.

### Business Insight

Employees experiencing slower career progression may become more likely to leave the organization.


#SECTION 6: SATISFACTION SCORES

In [18]:
query_11 = """
-- 6.1 Attrition rate by job satisfaction level
SELECT
    JobSatisfaction,
    CASE JobSatisfaction
        WHEN 1 THEN 'Low'
        WHEN 2 THEN 'Medium'
        WHEN 3 THEN 'High'
        WHEN 4 THEN 'Very High'
    END                                                                 AS satisfaction_label,
    COUNT(*)                                                            AS total,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY JobSatisfaction
ORDER BY JobSatisfaction;"""

df = pd.read_sql_query(query_11, conn)
df

,JobSatisfaction,satisfaction_label,total,attrition_rate_pct
0,1,Low,289,22.8
1,2,Medium,280,16.4
2,3,High,442,16.5
3,4,Very High,459,11.3


In [19]:
query_12 = """
-- 6.2 Work-life balance vs attrition
SELECT
    WorkLifeBalance,
    CASE WorkLifeBalance
        WHEN 1 THEN 'Bad'
        WHEN 2 THEN 'Good'
        WHEN 3 THEN 'Better'
        WHEN 4 THEN 'Best'
    END                                                                 AS wlb_label,
    COUNT(*)                                                            AS total,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
GROUP BY WorkLifeBalance
ORDER BY WorkLifeBalance; """

df = pd.read_sql_query(query_12, conn)
df

,WorkLifeBalance,wlb_label,total,attrition_rate_pct
0,1,Bad,80,31.3
1,2,Good,344,16.9
2,3,Better,893,14.2
3,4,Best,153,17.6


# Section 6 — Employee Satisfaction Analysis

This section explores how job satisfaction and work-life balance impact employee retention.

## 6.1 Job Satisfaction vs Attrition

This query analyzes attrition trends across different job satisfaction levels.

### Business Insight

Lower job satisfaction levels strongly correlate with increased employee attrition.

## 6.2 Work-Life Balance vs Attrition

This analysis evaluates how work-life balance influences workforce stability.

### Business Insight

Poor work-life balance is associated with higher attrition and may contribute to employee burnout.


#SECTION 7: FINANCIAL IMPACT

In [20]:
query_13 = """-- 7.1 Total estimated replacement cost (6x monthly salary = industry standard)
SELECT
    COUNT(*)                                                            AS employees_left,
    ROUND(AVG(MonthlyIncome), 2)                                        AS avg_monthly_salary,
    ROUND(AVG(MonthlyIncome) * 6, 2)                                    AS avg_replacement_cost,
    ROUND(SUM(MonthlyIncome * 6), 2)                                    AS total_replacement_cost
FROM employees
WHERE Attrition = 'Yes'; """

df = pd.read_sql_query(query_13, conn)
df

,employees_left,avg_monthly_salary,avg_replacement_cost,total_replacement_cost
0,237,4787.09,28722.56,6807246.0


In [21]:
query_14 = """ -- 7.2 Replacement cost breakdown by department
SELECT
    Department,
    COUNT(*)                                                            AS employees_left,
    ROUND(AVG(MonthlyIncome), 0)                                        AS avg_salary,
    ROUND(SUM(MonthlyIncome * 6), 0)                                    AS total_replacement_cost
FROM employees
WHERE Attrition = 'Yes'
GROUP BY Department
ORDER BY total_replacement_cost DESC;


"""

df = pd.read_sql_query(query_14, conn)
df

,Department,employees_left,avg_salary,total_replacement_cost
0,Research & Development,133,4108.0,3278244.0
1,Sales,92,5908.0,3261468.0
2,Human Resources,12,3716.0,267534.0


In [22]:
query_15 = """ -- 7.3 Top 10 most expensive attrition — by job role
SELECT
    JobRole,
    COUNT(*)                                                            AS employees_left,
    ROUND(AVG(MonthlyIncome), 0)                                        AS avg_salary,
    ROUND(SUM(MonthlyIncome * 6), 0)                                    AS total_replacement_cost
FROM employees
WHERE Attrition = 'Yes'
GROUP BY JobRole
ORDER BY total_replacement_cost DESC
LIMIT 10; """

df = pd.read_sql_query(query_15, conn)
df

,JobRole,employees_left,avg_salary,total_replacement_cost
0,Sales Executive,57,7489.0,2561238.0
1,Laboratory Technician,62,2919.0,1085964.0
2,Research Scientist,47,2780.0,784092.0
3,Manager,5,16797.0,503922.0
4,Sales Representative,33,2365.0,468216.0
5,Healthcare Representative,9,8548.0,461604.0
6,Manufacturing Director,10,7366.0,441930.0
7,Human Resources,12,3716.0,267534.0
8,Research Director,2,19396.0,232746.0


# Section 7 — Financial Impact of Attrition

This section estimates the business cost associated with employee turnover.

## 7.1 Estimated Total Replacement Cost

This query estimates the overall replacement cost of employees who left the organization.

### Business Insight

Employee attrition creates substantial financial losses due to hiring, onboarding, and training replacement employees.

## 7.2 Replacement Cost by Department

This analysis identifies departments generating the highest turnover-related replacement costs.

### Business Insight

Departments with high replacement costs may require immediate retention-focused strategies.

## 7.3 Most Expensive Job Role Attrition

This query identifies job roles with the highest financial impact from employee exits.

### Business Insight

Attrition among highly compensated or specialized roles creates disproportionately large business costs.


#SECTION 8: HIGH RISK EMPLOYEE PROFILE


In [23]:
query_16 = """ -- 8.1 Identify the highest-risk segment
--     Low salary + Overtime + Low satisfaction + Early tenure
SELECT
    COUNT(*)                                                            AS high_risk_employees,
    ROUND(AVG(MonthlyIncome), 0)                                        AS avg_salary,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                                   AS attrition_rate_pct
FROM employees
WHERE OverTime = 'Yes'
  AND MonthlyIncome < 5000
  AND JobSatisfaction <= 2
  AND YearsAtCompany <= 3; """

df = pd.read_sql_query(query_16, conn)
df

,high_risk_employees,avg_salary,attrition_rate_pct
0,39,2756.0,59.0


In [24]:
query_17 = """-- 8.2 Full high-risk employee list (for retention intervention)
SELECT
    EmployeeNumber,
    Department,
    JobRole,
    MonthlyIncome,
    OverTime,
    YearsAtCompany,
    JobSatisfaction,
    WorkLifeBalance,
    Attrition,
    ROUND(MonthlyIncome * 6, 0)                                         AS replacement_cost_if_left
FROM employees
WHERE OverTime = 'Yes'
  AND MonthlyIncome < 5000
  AND JobSatisfaction <= 2
  AND YearsAtCompany <= 3
ORDER BY MonthlyIncome ASC; """

df = pd.read_sql_query(query_17, conn)
df

,EmployeeNumber,Department,JobRole,MonthlyIncome,OverTime,YearsAtCompany,JobSatisfaction,WorkLifeBalance,Attrition,replacement_cost_if_left
0,811,Research & Development,Laboratory Technician,1601,Yes,0,1,3,Yes,9606.0
1,1248,Research & Development,Research Scientist,1859,Yes,1,2,4,Yes,11154.0
2,614,Sales,Sales Representative,1878,Yes,0,2,3,Yes,11268.0
3,1053,Research & Development,Research Scientist,2042,Yes,3,1,3,Yes,12252.0
4,1131,Research & Development,Research Scientist,2070,Yes,2,2,4,No,12420.0
5,133,Human Resources,Human Resources,2073,Yes,3,1,3,Yes,12438.0
6,1569,Research & Development,Laboratory Technician,2074,Yes,1,1,3,Yes,12444.0
7,959,Sales,Sales Representative,2121,Yes,1,2,4,Yes,12726.0
8,478,Sales,Sales Representative,2174,Yes,3,2,3,Yes,13044.0
9,1331,Sales,Sales Representative,2302,Yes,3,2,4,Yes,13812.0


# Section 8 — High-Risk Employee Identification

This section identifies employee groups with the highest probability of attrition based on multiple risk indicators.

## 8.1 High-Risk Segment Analysis

This query identifies employees with:

* Low salary
* Overtime workload
* Low job satisfaction
* Early tenure

### Business Insight

Employees matching multiple risk conditions represent the most vulnerable attrition segment within the organization.

## 8.2 High-Risk Employee List

This analysis generates a list of employees who may require proactive retention intervention.

### Business Insight

Identifying high-risk employees enables HR teams to take preventive actions before employee turnover occurs.


# Final Business Conclusion

The SQL analysis identified several major drivers of employee attrition, including overtime workload, low salary, low job satisfaction, and early organizational tenure.

Key findings include:

* Employees working overtime experience significantly higher attrition
* Lower salary bands show increased employee turnover
* Early-tenure employees represent the highest-risk retention group
* Poor work-life balance and low satisfaction strongly correlate with attrition
* Employee turnover creates substantial financial replacement costs

These insights can help organizations improve retention strategies, reduce workforce instability, and minimize turnover-related business expenses.
